# 🎤 super_Voz - StyleTTS2 Kaggle Pipeline

Este notebook executa o treinamento completo do StyleTTS2 no Kaggle seguindo a mesma lógica do notebook Colab: prepara o ambiente, clona/atualiza o repositório e executa o script real com logs em streaming.

### 🚀 Como usar no Kaggle:
1. Em **Settings**, ative **Internet** e selecione uma **GPU**.
2. Adicione seus áudios como Kaggle Dataset em `/kaggle/input/super-voz/Audios_brutos` ou `/kaggle/input/super-voz/Audios_Brutos`.
3. Crie o Kaggle Secret `HF_TOKEN` para sincronizar o pacote com `hf://buckets/warllem/Super_voz`.
4. Opcional: para upload TeraBox, crie Kaggle Secrets `TERABOX_NDUS`, `TERABOX_JS_TOKEN`, `TERABOX_CSRF_TOKEN`, `TERABOX_BROWSER_ID` e `TERABOX_NDUT_FMT`.
5. O TeraBox permanece opcional, mas a persistência padrão dos artefatos agora é o Hugging Face Bucket.
6. Durante o treino, checkpoints novos de época são enviados dentro do pacote `minha_voz_styletts2`.
7. Após upload confirmado, o working mantém o último checkpoint válido, `model/latest_checkpoint.pth` e `model/best_model.pth`.
8. Use **Save Version -> Save & Run All (Commit)** para deixar o Kaggle executando mesmo com o navegador fechado.

### Observação sobre Drive/TeraBox
Kaggle não oferece `google.colab.drive.mount()`. O TeraBox também não tem uma CLI oficial estável equivalente ao `rclone`; por isso esta integração usa comandos configuráveis e autenticação por cookie de sessão via Kaggle Secrets. O cookie `ndus` é uma credencial: não coloque esse valor no notebook nem no Git.

### Política Cloudflare
Este notebook preserva `cloudflare_r2` para permitir download dos áudios pelo Cloudflare R2, mas remove `output_prefix` e grava `disable_r2_uploads: true` na configuração runtime para bloquear upload/sync de checkpoints, dataset e resultados para o R2. A saída atual envia o pacote `minha_voz_styletts2` ao Hugging Face somente em checkpoint novo de época ou encerramento crítico, mantendo o último checkpoint válido e o melhor modelo conhecido.

### Se o Kaggle nao importar o notebook
Use o arquivo `run_kaggle_styletts2_celulas.md`: crie um notebook novo no Kaggle, copie as celulas na ordem indicada e depois execute **Run All**. Essa alternativa evita o erro da versao antiga antiga do bootstrap.


In [ ]:
# @title 🚀 INICIAR TUDO (ONE-CLICK)
from pathlib import Path
import os
import shutil
import subprocess
import sys
import threading
import time

try:
    from IPython.display import Javascript, display
except Exception:
    Javascript = None
    display = None

# Resemble Enhance fica ativo por padrao para reparar audios defeituosos antes da padronizacao final.
os.environ["SUPER_VOZ_ENABLE_RESEMBLE"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def start_kaggle_activity_watchdog(interval_seconds=90):
    """Mantem atividade visivel no notebook enquanto o pipeline roda."""
    stop_event = threading.Event()
    interval_ms = int(interval_seconds * 1000)

    if display is not None and Javascript is not None:
        display(Javascript(f"""
        (() => {{
          const key = '__superVozKaggleActivityWatchdog';
          if (window[key]) {{ clearInterval(window[key]); }}
          const tick = () => {{
            try {{
              const target = document.body || document.documentElement;
              const x = Math.floor(20 + Math.random() * 200);
              const y = Math.floor(20 + Math.random() * 200);
              target.dispatchEvent(new MouseEvent('mousemove', {{bubbles: true, clientX: x, clientY: y}}));
              window.dispatchEvent(new Event('focus'));
              console.log('[super_Voz] atividade Kaggle mantida', new Date().toISOString());
            }} catch (err) {{
              console.log('[super_Voz] watchdog Kaggle sem acesso ao DOM', err);
            }}
          }};
          tick();
          window[key] = setInterval(tick, {interval_ms});
        }})();
        """))

    def heartbeat():
        while not stop_event.wait(interval_seconds):
            stamp = time.strftime("%Y-%m-%d %H:%M:%S")
            print(f"[KAGGLE][ATIVIDADE] {stamp} - pipeline ativo; mantendo saida do notebook em movimento.", flush=True)

    thread = threading.Thread(target=heartbeat, name="kaggle-activity-watchdog", daemon=True)
    thread.start()
    print(f"[KAGGLE][ATIVIDADE] Watchdog iniciado a cada {interval_seconds}s.", flush=True)
    return stop_event


def stop_kaggle_activity_watchdog(stop_event):
    if stop_event is not None:
        stop_event.set()
    if display is not None and Javascript is not None:
        display(Javascript("""
        (() => {
          const key = '__superVozKaggleActivityWatchdog';
          if (window[key]) { clearInterval(window[key]); delete window[key]; }
        })();
        """))


def load_kaggle_secret_to_env(secret_name="TERABOX_NDUS", env_name=None, required=False):
    env_name = env_name or secret_name
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(secret_name)
    except Exception as exc:
        if required:
            print(f"Secret obrigatorio {secret_name} nao encontrado ou indisponivel ({exc}).")
        return False

    if value:
        os.environ[env_name] = value
        print(f"Secret {secret_name} carregado para {env_name}.")
        return True
    return False


def load_runtime_secrets():
    secret_map = {
        "HF_TOKEN": ("HF_TOKEN", False),
        "TERABOX_NDUS": ("TERABOX_NDUS", False),
        "TERABOX_JS_TOKEN": ("TERABOX_JS_TOKEN", False),
        "TERABOX_CSRF_TOKEN": ("TERABOX_CSRF_TOKEN", False),
        "TERABOX_BROWSER_ID": ("TERABOX_BROWSER_ID", False),
        "TERABOX_NDUT_FMT": ("TERABOX_NDUT_FMT", False),
        "R2_ENDPOINT_URL": ("R2_ENDPOINT_URL", False),
        "R2_BUCKET_NAME": ("R2_BUCKET_NAME", False),
        "R2_ACCESS_KEY_ID": ("R2_ACCESS_KEY_ID", True),
        "R2_SECRET_ACCESS_KEY": ("R2_SECRET_ACCESS_KEY", True),
        "R2_RAW_AUDIO_PREFIX": ("R2_RAW_AUDIO_PREFIX", False),
    }
    for secret_name, (env_name, required) in secret_map.items():
        load_kaggle_secret_to_env(secret_name, env_name, required=required)


RUNNER_RELATIVE_PATH = Path("super_Voz") / "kaglle" / "scripts" / "run_kaggle_styletts2.py"


def has_kaggle_runner(repo_root):
    return (repo_root / RUNNER_RELATIVE_PATH).exists()


def find_kaggle_input_repo():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for runner in input_root.rglob(str(RUNNER_RELATIVE_PATH)):
        return runner.parents[3]
    return None


def locate_project_dir(repo_base_dir):
    preferred_dir = repo_base_dir / "super_Voz" / "kaglle"
    preferred_runner = preferred_dir / "scripts" / "run_kaggle_styletts2.py"
    if preferred_runner.exists():
        return preferred_dir.resolve()

    target_script = "run_kaggle_styletts2.py"
    found_path = None
    for path in repo_base_dir.rglob(target_script):
        if "kaglle" in path.parts and path.parent.name == "scripts":
            found_path = path.parent.parent
            break
    if found_path is not None:
        return found_path.resolve()

    found_scripts = []
    if repo_base_dir.exists():
        found_scripts = sorted(str(p.relative_to(repo_base_dir)) for p in repo_base_dir.rglob(target_script))
    raise FileNotFoundError(
        f"Runner Kaggle nao encontrado em {preferred_runner}. "
        f"Encontrados: {found_scripts or 'nenhum'}"
    )


def ensure_project_root(repo_url, repo_base_dir):
    if repo_base_dir.exists():
        subprocess.run(["git", "-C", str(repo_base_dir), "remote", "set-url", "origin", repo_url], check=False)
        fetch = subprocess.run(["git", "-C", str(repo_base_dir), "fetch", "--all"], check=False)
        if fetch.returncode == 0:
            subprocess.run(["git", "-C", str(repo_base_dir), "checkout", "main"], check=False)
            subprocess.run(["git", "-C", str(repo_base_dir), "reset", "--hard", "origin/main"], check=False)
        else:
            print("[AVISO] GitHub indisponivel; usando clone local existente se estiver valido.")
        if has_kaggle_runner(repo_base_dir):
            return repo_base_dir
    else:
        try:
            subprocess.run(["git", "clone", repo_url, str(repo_base_dir)], check=True)
        except subprocess.CalledProcessError as exc:
            print(f"[AVISO] Clone via GitHub falhou ({exc}). Tentando Kaggle Input local.")
        if has_kaggle_runner(repo_base_dir):
            return repo_base_dir

    input_repo = find_kaggle_input_repo()
    if input_repo:
        print(f"--- Usando copia anexada em Kaggle Input: {input_repo} ---")
        shutil.copytree(input_repo, repo_base_dir, dirs_exist_ok=True)
        if has_kaggle_runner(repo_base_dir):
            return repo_base_dir

    raise RuntimeError(
        "Nao foi possivel obter o codigo do super_Voz. "
        "Ative Internet no Kaggle para clonar o GitHub ou anexe um Kaggle Dataset "
        "contendo super_Voz/kaglle/scripts/run_kaggle_styletts2.py."
    )


def setup_environment():
    repo_url = "https://github.com/warllemedicao/voz_stylle.git"
    repo_base_dir = Path("/kaggle/working/Super_voz")

    print(f"--- 1. Clonando/Atualizando Repositório: {repo_url} ---")
    ensure_project_root(repo_url, repo_base_dir)

    print("--- 2. Localizando diretório Kaggle do projeto ---")
    project_dir = locate_project_dir(repo_base_dir)
    os.chdir(project_dir)
    print(f"Projeto: {project_dir}")

    print("--- 3. Instalando dependências críticas ---")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "boto3"], check=True)

    print("--- 4. Verificando Hardware e Aceleração ---")
    try:
        import torch
        if torch.cuda.is_available():
            print(f"GPU Detectada: {torch.cuda.get_device_name(0)}")
        else:
            print("AVISO: GPU não detectada. Ative GPU em Settings antes de rodar o treino.")
    except Exception as exc:
        print(f"Verificação de GPU adiada para o pipeline: {exc}")

    print("--- 5. Política de persistência ---")
    print("Kaggle não monta Google Drive como Colab. Upload/sync para Cloudflare R2 fica desligado nesta execução para evitar taxas.")
    print("Download de áudios do Cloudflare R2 continua permitido se cloudflare_r2 estiver configurado.")
    print("TeraBox será usado somente se os secrets TERABOX_* de sessão estiverem configurados.")

    return project_dir


def make_no_cloud_config(project_dir: Path) -> str:
    import yaml

    source_config = project_dir / "styletts2_kaggle_config.yml"
    runtime_config = project_dir / "styletts2_kaggle_sem_cloudflare.yml"
    with source_config.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}

    r2_cfg = cfg.get("cloudflare_r2", {}) or {}
    r2_cfg.pop("output_prefix", None)
    r2_cfg["disable_r2_uploads"] = True
    cfg["cloudflare_r2"] = r2_cfg
    cfg["raw_audio_candidates"] = [
        "Audios_brutos",
        "/kaggle/input/super-voz/Audios_brutos",
        "/kaggle/input/super-voz/Audios_Brutos",
    ]
    cfg["styletts2_restore_candidates"] = [
        "/kaggle/input/styllet2",
        "/kaggle/input/styletts2",
        "/kaggle/input/terabox/StyleTTS2",
        "/kaggle/input/super-voz/StyleTTS2",
    ]
    cfg["output_dir"] = "/kaggle/working/super_Voz_outputs"

    hf_cfg = cfg.get("huggingface", {}) or {}
    hf_cfg["enabled"] = bool(hf_cfg.get("enabled", True) and hf_cfg.get("bucket_uri"))
    cfg["huggingface"] = hf_cfg
    print("Config runtime: Hugging Face Bucket obrigatorio; o runner validara HF_TOKEN antes do treino.")

    tb_cfg = cfg.get("terabox", {}) or {}
    tb_cfg["enabled"] = bool(tb_cfg.get("enabled") or os.environ.get(tb_cfg.get("ndus_env", "TERABOX_NDUS")))
    cfg["terabox"] = tb_cfg
    if tb_cfg["enabled"]:
        print("Config runtime: TeraBox ativado.")
        missing_tb = [name for name in tb_cfg.get("required_env", []) if not os.environ.get(name)]
        if missing_tb:
            print(f"AVISO: TeraBox sem todos os secrets: {', '.join(missing_tb)}")
    else:
        print("Config runtime: TeraBox autenticado desativado; sera tentada restauracao via Kaggle Input styllet2/styletts2.")

    with runtime_config.open("w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

    print(f"Config runtime com download R2 permitido e upload R2 bloqueado: {runtime_config}")
    return runtime_config.name


def package_outputs() -> Path:
    package_dir = Path("/kaggle/working/StyleTTS2/minha_voz_styletts2")
    print(f"Pacote local da voz: {package_dir}")
    print("O pacote tambem foi sincronizado com hf://buckets/warllem/Super_voz quando HF_TOKEN estava configurado.")
    return package_dir


PROJECT_DIR = setup_environment()
os.chdir(PROJECT_DIR)
load_runtime_secrets()

CONFIG_NAME = make_no_cloud_config(PROJECT_DIR)
cmd = [sys.executable, "-u", "scripts/run_kaggle_styletts2.py", "--config", CONFIG_NAME]

print("="*60)
print("INICIANDO PIPELINE SUPER_VOZ")
print("="*60)
print(f"Comando: {' '.join(cmd)}")

activity_stop = start_kaggle_activity_watchdog(interval_seconds=90)
try:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")

    code = proc.wait()
    if code != 0:
        print(f"\n[ERRO] O pipeline falhou com código {code}.")
    else:
        print("\nPipeline finalizado com sucesso!")
finally:
    stop_kaggle_activity_watchdog(activity_stop)

package_outputs()
